# KADMON Optuna — Binning + Partial OT

Le classement utilise `global_distance_mm`. Pour chaque bundle, l'objectif Optuna minimise le rapport entre la distance intra-identité et la distance moyenne inter-identité.


## 1. Imports et configuration


In [18]:
from pathlib import Path
import sys
from time import perf_counter

import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from optuna.trial import TrialState


NOTEBOOK_DIR = Path.cwd().resolve()
KADMON_ROOT = (NOTEBOOK_DIR / "../..").resolve()
BUNDLES_DIR = NOTEBOOK_DIR.parent / "bundles"
STUDY_PATH = NOTEBOOK_DIR / "studies" / "optuna_reid.sqlite3"
if str(KADMON_ROOT) not in sys.path:
    sys.path.insert(0, str(KADMON_ROOT))

from kadmon.comparison import compare_bundles

optuna.logging.set_verbosity(optuna.logging.WARNING)

EXPERIMENT_NAME = "binning_partial"
COMPRESSION = "binning"
TRANSPORT = "partial"
N_TRIALS = 120


## 2. Données

Les bundles sont chargés depuis le dossier frère `../bundles`.


In [19]:
SUBJECT_PAIRS = {
    "103818": "103818_re",
    "135528": "135528_re",
    "143325": "143325_re",
    "177746": "177746_re",
    "194140": "194140_re",
    "250427": "250427_re",
    "433839": "433839_re",
    "627549": "627549_re",
    "783462": "783462_re",
    "861456": "861456_re",
}

REFERENCE_SUBJECT = "103818"
INTRA_IDENTITY_SUBJECT = SUBJECT_PAIRS[REFERENCE_SUBJECT]
COMPARISON_SUBJECTS = tuple(SUBJECT_PAIRS.values())
SUBJECTS = (REFERENCE_SUBJECT, *COMPARISON_SUBJECTS)
N_POINTS = 12
SEED = 42
BUNDLES_TO_RUN = None  # tuple de noms exacts pour un essai court, sinon None
# Partial OT construit des matrices denses : rester séquentiel évite de
# multiplier leur mémoire par le nombre de workers.
N_JOBS_BUNDLES = 1
MAX_COST_MATRIX_BYTES = 512 * 2**20  # 512 Mio estimés pour la comparaison dense

def index_subject(subject):
    directory = BUNDLES_DIR / subject / "nn_8mm"
    suffix = f"_{N_POINTS}mpts_rasmm.npy"
    paths = sorted(directory.glob(f"*{suffix}"))
    return {path.name[:-len(suffix)]: path for path in paths}


SUBJECT_FILES = {subject: index_subject(subject) for subject in SUBJECTS}
bundle_names = sorted(set.intersection(*(set(index) for index in SUBJECT_FILES.values())))
if BUNDLES_TO_RUN is not None:
    missing = sorted(set(BUNDLES_TO_RUN) - set(bundle_names))
    if missing:
        raise ValueError(f"Bundles demandés absents : {missing}")
    bundle_names = [name for name in bundle_names if name in BUNDLES_TO_RUN]
if not bundle_names:
    raise RuntimeError("Aucun bundle commun au protocole de ré-identification.")

bundle_cache = {}
# Le cache inclut aussi le type et tous les paramètres de compression via kadmon.
compression_cache = {}


def load_bundle(subject, bundle_name):
    key = (subject, bundle_name)
    if key not in bundle_cache:
        bundle = np.load(SUBJECT_FILES[subject][bundle_name], mmap_mode="r")
        expected_shape = (N_POINTS, 3)
        if (
            bundle.ndim != 3
            or bundle.shape[1:] != expected_shape
            or len(bundle) == 0
            or not np.isfinite(bundle).all()
        ):
            raise ValueError(
                f"Bundle invalide : {subject}/{bundle_name}, forme={bundle.shape}"
            )
        bundle_cache[key] = bundle
    return bundle_cache[key]


print(f"Données : {BUNDLES_DIR}")
print("Protocole : 1 comparaison intra-identité et 9 comparaisons inter-identité par bundle")
print(f"Bundles communs : {len(bundle_names)}")
display(pd.DataFrame({"bundle": bundle_names}))


Données : /home/colin/Tractographie/KADMON/notebooks/bundles
Protocole : 1 comparaison intra-identité et 9 comparaisons inter-identité par bundle
Bundles communs : 31


,bundle
0,tractosearch_nn_8_0mm_all_AF_L_m
1,tractosearch_nn_8_0mm_all_AF_R_m
2,tractosearch_nn_8_0mm_all_CC_1_m
3,tractosearch_nn_8_0mm_all_CC_2a_m
4,tractosearch_nn_8_0mm_all_CC_2b_m
5,tractosearch_nn_8_0mm_all_CC_3_m
6,tractosearch_nn_8_0mm_all_CC_4_m
7,tractosearch_nn_8_0mm_all_CC_5_m
8,tractosearch_nn_8_0mm_all_CC_6_m
9,tractosearch_nn_8_0mm_all_CC_7_m


## 3. Espace de recherche

- `bin_size` : 2 à 16 mm, par pas de 1 mm
- `binning_nb` : 2 ou 3 (seules valeurs implémentées par tractosearch)
- `method` : `mean` ou `median`
- `mass` : 0,50 à 1,00, par pas de 0,01

`n_points=12` reste fixe.

> **Note sur les points réservoirs (`nb_dummies=100`).** Partial OT ajoute des
> points artificiels pour recevoir la fraction de masse qui n'est pas transportée
> (par exemple 16 % si `mass=0.84`). Le binning peut produire des milliers de
> représentants aux poids très petits et irréguliers; concentrer toute cette masse
> sur l'unique dummy utilisé par défaut dans POT peut alors déstabiliser le solveur
> EMD. La répartir sur 100 dummies améliore sa stabilité. Ce problème apparaît
> beaucoup moins avec K-means, qui impose peu de centroïdes aux poids plus élevés.
> Ces points sont purement numériques et sont retirés du plan avant le calcul des
> métriques : ils ne représentent pas de vraies fibres.


In [20]:
def sample_parameters(trial):
    return (
        {
            "bin_size": trial.suggest_float("bin_size", 2.0, 16.0, step=1.0),
            "binning_nb": trial.suggest_int("binning_nb", 2, 3),
            "method": trial.suggest_categorical("method", ["mean", "median"]),
            "n_points": N_POINTS,
        },
        {
            "mass": trial.suggest_float("mass", 0.50, 1.00, step=0.01),
            "nb_dummies": 100,
        },
    )


## 4. Objectif Optuna

La compression, MDF, le transport et les statistiques sont calculés par `compare_bundles()`.


In [21]:
def evaluate_bundle(bundle_name, compression_parameters, transport_parameters):
    source = load_bundle(REFERENCE_SUBJECT, bundle_name)

    pair_metrics = []
    for candidate_subject in COMPARISON_SUBJECTS:
        target = load_bundle(candidate_subject, bundle_name)
        try:
            result = compare_bundles(
                source,
                target,
                compression=COMPRESSION,
                transport=TRANSPORT,
                compression_parameters=compression_parameters,
                transport_parameters=transport_parameters,
                compression_cache=compression_cache,
                source_compression_key=(REFERENCE_SUBJECT, bundle_name),
                target_compression_key=(candidate_subject, bundle_name),
                max_cost_matrix_bytes=MAX_COST_MATRIX_BYTES,
            )
        except MemoryError as exc:
            raise optuna.TrialPruned(f"Configuration trop volumineuse : {exc}") from exc
        except ValueError as exc:
            if "Error in the EMD resolution" in str(exc):
                raise optuna.TrialPruned(f"EMD instable : {exc}") from exc
            raise
        metrics = result["metrics"]
        pair_metrics.append({
            "global_distance_mm": float(metrics["global_distance_mm"]),
            "mean_displacement_mm": float(metrics["mean_mm"]),
            "transported_mass": float(metrics["transported_mass"]),
            "source_n_representatives": int(metrics["source_n_representatives"]),
            "target_n_representatives": int(metrics["target_n_representatives"]),
        })

    # La compression de la référence doit être reproductible pour tous les candidats.
    if len({row["source_n_representatives"] for row in pair_metrics}) != 1:
        raise RuntimeError(f"Compression source non reproductible pour {bundle_name}.")

    distances = np.asarray(
        [row["global_distance_mm"] for row in pair_metrics], dtype=np.float64
    )
    intra_index = COMPARISON_SUBJECTS.index(INTRA_IDENTITY_SUBJECT)
    intra_distance = float(distances[intra_index])
    inter_distances = np.delete(distances, intra_index)
    mean_inter_distance = float(inter_distances.mean())
    if not np.isfinite(mean_inter_distance) or mean_inter_distance <= 0:
        raise RuntimeError(
            f"Distance moyenne inter-identité invalide : {mean_inter_distance}"
        )

    order = np.argsort(distances, kind="stable")
    return {
        "bundle": bundle_name,
        "intra_inter_ratio": intra_distance / mean_inter_distance,
        "intra_identity_top1_success": bool(int(np.argmin(distances)) == intra_index),
        "intra_identity_rank": int(np.flatnonzero(order == intra_index)[0] + 1),
        "intra_inter_separation_margin_mm": float(inter_distances.min() - intra_distance),
        "intra_identity_distance_mm": intra_distance,
        "mean_inter_identity_distance_mm": mean_inter_distance,
        "mean_displacement_mm": float(np.mean([row["mean_displacement_mm"] for row in pair_metrics])),
        "mean_transported_mass": float(np.mean([row["transported_mass"] for row in pair_metrics])),
        "mean_n_representatives": float(np.mean([
            value
            for row in pair_metrics
            for value in (row["source_n_representatives"], row["target_n_representatives"])
        ])),
    }


def objective(trial):
    started = perf_counter()
    compression_parameters, transport_parameters = sample_parameters(trial)
    tasks = (
        delayed(evaluate_bundle)(
            bundle_name, compression_parameters, transport_parameters
        )
        for bundle_name in bundle_names
    )
    results = (
        [
            evaluate_bundle(
                bundle_name, compression_parameters, transport_parameters
            )
            for bundle_name in bundle_names
        ]
        if N_JOBS_BUNDLES == 1
        else Parallel(
            n_jobs=min(N_JOBS_BUNDLES, len(bundle_names)), backend="threading"
        )(tasks)
    )
    bundle_metrics = pd.DataFrame(results)
    valid_mask = bundle_metrics["intra_identity_top1_success"]

    aggregates = {
        "reid_valid_bundles": bundle_metrics.loc[valid_mask, "bundle"].tolist(),
        "reid_failed_bundles": bundle_metrics.loc[~valid_mask, "bundle"].tolist(),
        "mean_intra_inter_ratio": float(bundle_metrics["intra_inter_ratio"].mean()),
        "median_intra_inter_ratio": float(bundle_metrics["intra_inter_ratio"].median()),
        "intra_identity_top1_accuracy": float(bundle_metrics["intra_identity_top1_success"].mean()),
        "mean_intra_identity_rank": float(bundle_metrics["intra_identity_rank"].mean()),
        "mean_intra_inter_separation_margin_mm": float(bundle_metrics["intra_inter_separation_margin_mm"].mean()),
        "mean_intra_identity_distance_mm": float(bundle_metrics["intra_identity_distance_mm"].mean()),
        "mean_inter_identity_distance_mm": float(bundle_metrics["mean_inter_identity_distance_mm"].mean()),
        "mean_displacement_mm": float(bundle_metrics["mean_displacement_mm"].mean()),
        "mean_transported_mass": float(bundle_metrics["mean_transported_mass"].mean()),
        "mean_n_representatives": float(bundle_metrics["mean_n_representatives"].mean()),
        "n_bundles": int(len(bundle_metrics)),
        "n_comparisons": int(len(bundle_metrics) * len(COMPARISON_SUBJECTS)),
        "elapsed_s": float(perf_counter() - started),
    }
    for name, value in aggregates.items():
        trial.set_user_attr(name, value)
    return aggregates["mean_intra_inter_ratio"]


## 5. Optimisation

La base SQLite est l'unique sortie automatique de l'étude.


In [22]:
STUDY_PATH.parent.mkdir(parents=True, exist_ok=True)

study = optuna.create_study(
    study_name=EXPERIMENT_NAME,
    storage=f"sqlite:///{STUDY_PATH}",
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

completed_trials = sum(trial.state == TrialState.COMPLETE for trial in study.trials)
remaining_trials = max(0, N_TRIALS - completed_trials)
print(
    f"Étude : {completed_trials} essais terminés, "
    f"{remaining_trials} essais à exécuter."
)
if remaining_trials:
    study.optimize(
        objective,
        n_trials=remaining_trials,
        show_progress_bar=True,
        gc_after_trial=True,
        callbacks=[lambda study, trial: compression_cache.clear()],
    )

print(f"Base SQLite : {STUDY_PATH}")


Étude : 120 essais terminés, 0 essais à exécuter.
Base SQLite : /home/colin/Tractographie/KADMON/notebooks/optuna/studies/optuna_reid.sqlite3


## 6. Analyse des résultats


In [23]:
trials_df = study.trials_dataframe(
    attrs=("number", "value", "params", "user_attrs", "state")
)
display(trials_df.sort_values("value").head(10))

best = study.best_trial
best_summary = pd.Series({
    "experiment": EXPERIMENT_NAME,
    "trial": best.number,
    "mean_intra_inter_ratio": best.value,
    **best.params,
    **best.user_attrs,
}, name="meilleur essai")
display(best_summary.to_frame())


,number,value,params_bin_size,params_binning_nb,params_mass,params_method,user_attrs_elapsed_s,user_attrs_intra_identity_top1_accuracy,user_attrs_mean_displacement_mm,user_attrs_mean_inter_identity_distance_mm,...,user_attrs_mean_intra_inter_ratio,user_attrs_mean_intra_inter_separation_margin_mm,user_attrs_mean_n_representatives,user_attrs_mean_transported_mass,user_attrs_median_intra_inter_ratio,user_attrs_n_bundles,user_attrs_n_comparisons,user_attrs_reid_failed_bundles,user_attrs_reid_valid_bundles,state
108,108,0.314944,16.0,2,0.5,mean,591.916731,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
109,109,0.314944,16.0,2,0.5,mean,590.925114,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
120,120,0.314944,16.0,2,0.5,mean,595.912299,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
112,112,0.314944,16.0,2,0.5,mean,595.657399,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
118,118,0.314944,16.0,2,0.5,mean,592.255895,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
100,100,0.314944,16.0,2,0.5,mean,594.499368,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
89,89,0.314944,16.0,2,0.5,mean,594.237078,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
140,140,0.314944,16.0,2,0.5,mean,615.804627,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
137,137,0.314944,16.0,2,0.5,mean,594.928198,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
138,138,0.314944,16.0,2,0.5,mean,593.313063,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE


,meilleur essai
experiment,binning_partial
trial,89
mean_intra_inter_ratio,0.314944
bin_size,16.0
binning_nb,2
method,mean
mass,0.5
elapsed_s,594.237078
intra_identity_top1_accuracy,0.967742
mean_displacement_mm,3.589178


## 7. Visualisations Optuna

Les importances donnent d'abord une vue globale, puis les coupes montrent l'effet individuel de chaque paramètre. Le contour est limité à `bin_size` et `mass`, les deux paramètres continus pour lesquels une surface 2D est réellement lisible.


In [28]:
from plotly.io import show

fig_importance = optuna.visualization.plot_param_importances(study)
show(fig_importance)

fig_slice = optuna.visualization.plot_slice(study)
show(fig_slice)


In [25]:
df = study.trials_dataframe()[[
    "number", "value", "state", "params_mass", "params_bin_size",
    "params_binning_nb", "params_method",
    "user_attrs_mean_n_representatives",
    "user_attrs_intra_identity_top1_accuracy",
]]
df = df[df["state"] == "COMPLETE"].dropna(subset=["value"]).rename(columns={
    "number": "trial", "value": "score",
    "params_mass": "mass", "params_bin_size": "bin_size",
    "params_binning_nb": "binning_nb", "params_method": "method",
    "user_attrs_mean_n_representatives": "n_representatives",
    "user_attrs_intra_identity_top1_accuracy": "reid_accuracy",
})
best_score = df["score"].min()

rows, selected_trials = [], set()
for threshold in (1, 2, 5):
    candidates = df[
        (df["score"] <= best_score * (1 + threshold / 100))
        & ~df["trial"].isin(selected_trials)
    ]
    # Ne pas répéter un trial déjà choisi; parmi les solutions restantes,
    # maximiser mass conserve davantage du faisceau.
    row = candidates.sort_values(
        ["mass", "score"], ascending=[False, True]
    ).iloc[0]
    selected_trials.add(int(row.trial))
    rows.append([
        threshold, int(row.trial), row.score,
        100 * (row.score / best_score - 1), row.mass, row.bin_size,
        int(row.binning_nb), row.method, row.n_representatives,
        row.reid_accuracy,
    ])

tradeoff = pd.DataFrame(rows, columns=[
    "seuil (%)", "trial", "score", "écart relatif (%)", "mass",
    "bin_size", "binning_nb", "method", "n_representatives",
    "reid_accuracy",
])
display(tradeoff.style.format({
    "score": "{:.6f}", "écart relatif (%)": "{:.2f}",
    "mass": "{:.2f}", "bin_size": "{:.0f}",
    "n_representatives": "{:.0f}", "reid_accuracy": "{:.0%}",
}))


,seuil (%),trial,score,écart relatif (%),mass,bin_size,binning_nb,method,n_representatives,reid_accuracy
0,1,55,0.317031,0.66,0.68,16,2,mean,147,97%
1,2,56,0.316408,0.47,0.61,16,2,mean,147,97%
2,5,57,0.316272,0.42,0.60,16,2,mean,147,97%


## Interprétation des compromis observés

L'étude finale contient **120 essais `COMPLETE`**. Elle compte aussi 14 essais `PRUNED`, 5 essais `FAIL` et 2 anciens essais `RUNNING`; seuls les essais `COMPLETE` sont utilisés pour le classement.

Le **trial 89** est l'optimum strict observé (`score=0,314944`, `bin_size=16 mm`, `binning_nb=2`, `method=mean`, `mass=0,50`). La compression produit en moyenne **147 représentants** par distribution. Son exactitude RE-ID Top-1 est de **96,77 %** (30 bundles sur 31); le seul échec est `CST_L`.

Le **trial 55** constitue le meilleur compromis proche de l'optimum : `score=0,317031`, `bin_size=16 mm`, `binning_nb=2`, `method=mean`, `mass=0,68`, soit seulement **+0,66 %** sur le score. Il conserve 36 % de masse transportée supplémentaire, avec la même exactitude Top-1 de 96,77 % et le même échec `CST_L`. Le tableau 1–2–5 % exclut maintenant les trials déjà sélectionnés afin de présenter trois alternatives distinctes. Dans cette étude, même les alternatives affichées aux seuils 2 % et 5 % se trouvent déjà à moins de 1 % de l'optimum; il n'existe aucun essai `COMPLETE` dans la bande 2–5 %.

- Selon la règle RE-ID commune, le **trial 121** est le meilleur classement (`bin_size=8 mm`, `binning_nb=2`, `method=mean`, `mass=0,50`) : Top-1 100 %, rang moyen 1,00 et environ 996 représentants.
- Pour l'usage anatomique de KADMON, le projet utilise le **trial 55 par défaut** : `mass=0,68`, même Top-1 que l'optimum strict local à 16 mm et couverture supérieure.
- Pour une future initialisation **LDDMM**, privilégier également le trial 55, car il transporte une fraction plus importante du faisceau sans perte d'exactitude Top-1.
- L'optimum de `bin_size` se trouve à la borne supérieure testée (16 mm). Cela indique qu'une compression plus forte stabilise la RE-ID dans cet espace, mais justifie un contrôle ciblé au-delà de 16 mm avant de conclure que 16 mm est l'optimum réel.
- Contrairement à `epsilon` dans Sinkhorn, `mass` représente ici une **couverture anatomique explicite** : une masse faible améliore parfois le score en ignorant davantage de streamlines, tandis qu'une masse élevée produit une comparaison plus complète du faisceau.
